# 📊 Comprehensive Model Evaluation

> **Master evaluation metrics and validation techniques for ML models**

This notebook provides comprehensive coverage of model evaluation techniques, metrics, and best practices for machine learning projects.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Master** all major evaluation metrics for classification and regression
- **Understand** cross-validation and its variants
- **Implement** proper evaluation frameworks
- **Handle** imbalanced datasets appropriately
- **Create** comprehensive evaluation reports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✅ All imports successful!")

## 📊 Classification Metrics Deep Dive

In [ ]:
class ComprehensiveEvaluator:
    def __init__(self, model, X_test, y_test, class_names=None):
        self.model = model
        self.X_test = X_test
        self.y_test = y_test
        self.class_names = class_names or ['Class 0', 'Class 1']
        
        # Generate predictions
        self.y_pred = model.predict(X_test)
        if hasattr(model, 'predict_proba'):
            self.y_pred_proba = model.predict_proba(X_test)[:, 1]
        else:
            self.y_pred_proba = None
    
    def classification_report_detailed(self):
        """Generate detailed classification report"""
        print("=== COMPREHENSIVE CLASSIFICATION REPORT ===")
        
        # Basic metrics
        accuracy = accuracy_score(self.y_test, self.y_pred)
        precision = precision_score(self.y_test, self.y_pred, average='weighted')
        recall = recall_score(self.y_test, self.y_pred, average='weighted')
        f1 = f1_score(self.y_test, self.y_pred, average='weighted')
        
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision (weighted): {precision:.4f}")
        print(f"Recall (weighted): {recall:.4f}")
        print(f"F1-Score (weighted): {f1:.4f}")
        
        if self.y_pred_proba is not None:
            auc = roc_auc_score(self.y_test, self.y_pred_proba)
            print(f"ROC AUC: {auc:.4f}")
        
        print("\n=== DETAILED CLASSIFICATION REPORT ===")
        print(classification_report(self.y_test, self.y_pred, 
                                  target_names=self.class_names))
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'auc': auc if self.y_pred_proba is not None else None
        }
    
    def plot_confusion_matrix(self):
        """Plot confusion matrix with detailed annotations"""
        cm = confusion_matrix(self.y_test, self.y_pred)
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=self.class_names, yticklabels=self.class_names)
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        
        # Add percentage annotations
        total = cm.sum()
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j+0.5, i+0.7, f'({cm[i,j]/total:.1%})', 
                        ha='center', va='center', fontsize=10)
        
        plt.tight_layout()
        plt.show()
    
    def plot_roc_curve(self):
        """Plot ROC curve with AUC"""
        if self.y_pred_proba is None:
            print("No probability predictions available for ROC curve")
            return
        
        fpr, tpr, thresholds = roc_curve(self.y_test, self.y_pred_proba)
        auc_score = auc(fpr, tpr)
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, 
                label=f'ROC curve (AUC = {auc_score:.3f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic (ROC) Curve')
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.show()
    
    def full_evaluation_report(self):
        """Generate complete evaluation report"""
        metrics = self.classification_report_detailed()
        self.plot_confusion_matrix()
        self.plot_roc_curve()
        return metrics

print("✅ ComprehensiveEvaluator class defined!")

In [ ]:
# Generate sample data and train model
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, 
                          n_redundant=2, n_clusters_per_class=1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Comprehensive evaluation
evaluator = ComprehensiveEvaluator(model, X_test, y_test, ['Negative', 'Positive'])
metrics = evaluator.full_evaluation_report()

## 🔄 Cross-Validation Framework

In [ ]:
def comprehensive_cross_validation(model, X, y, cv_folds=5):
    """
    Perform comprehensive cross-validation evaluation
    """
    # Define scoring metrics
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    
    # Stratified K-Fold for balanced splits
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    
    results = {}
    
    for metric in scoring:
        scores = cross_val_score(model, X, y, cv=skf, scoring=metric)
        results[metric] = {
            'mean': scores.mean(),
            'std': scores.std(),
            'scores': scores
        }
    
    # Print results
    print("=== CROSS-VALIDATION RESULTS ===")
    for metric, stats in results.items():
        print(f"{metric.upper()}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    
    # Plot results
    fig, ax = plt.subplots(figsize=(12, 6))
    
    metrics_names = list(results.keys())
    means = [results[m]['mean'] for m in metrics_names]
    stds = [results[m]['std'] for m in metrics_names]
    
    ax.bar(metrics_names, means, yerr=stds, capsize=5, alpha=0.7)
    ax.set_ylabel('Score')
    ax.set_title('Cross-Validation Results')
    ax.set_ylim(0, 1)
    
    # Add value labels on bars
    for i, (mean, std) in enumerate(zip(means, stds)):
        ax.text(i, mean + std + 0.01, f'{mean:.3f}', 
               ha='center', va='bottom', fontweight='bold')
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    return results

# Test cross-validation
cv_results = comprehensive_cross_validation(model, X_train, y_train)

## 🎯 Practice Problems

### **Problem 1: Imbalanced Dataset Evaluation**
Create evaluation framework for imbalanced datasets.

In [ ]:
def evaluate_imbalanced_dataset(model, X_test, y_test):
    """
    Comprehensive evaluation for imbalanced datasets
    
    Include:
    - Precision-Recall curve
    - Balanced accuracy
    - Matthews Correlation Coefficient
    - Class-specific metrics
    
    Returns:
    dict: Comprehensive metrics
    """
    # Your code here
    pass

# Test with imbalanced data
# X_imb, y_imb = make_classification(n_samples=1000, weights=[0.9, 0.1], random_state=42)
# results = evaluate_imbalanced_dataset(model, X_imb, y_imb)

## 🎯 Key Takeaways

1. **Multiple metrics** provide different perspectives on model performance
2. **Cross-validation** gives robust performance estimates
3. **Confusion matrices** reveal specific error patterns
4. **ROC curves** help with threshold selection
5. **Imbalanced datasets** require special evaluation considerations

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Apply to real datasets** with different characteristics
3. **Move to the next notebook**: Feature Engineering

---

**Excellent evaluation skills!** 🎉 You can now properly assess model performance.